# Aff-Wild2 Stage 1 — train, eval, smoothing grid

For each of the two backbones (`enet_b0_8_va_mtl`, `mbf_va_mtl`):
1. Train three heads sequentially (EXPR → VA → AU) via `src/train.py` on the cached features.
2. Raw evaluation on the validation split.
3. Per-video Gaussian smoothing grid over EXPR + VA (never AU). Grid matches Paper A cell 66.
4. Aggregate a comparison table against Paper A cell 60 'aligned' targets into `results/aw2_stage1/summary.md`.

Prerequisite: `aw2_01_extract_visual.ipynb` has populated both `cache/features/{enet_b0_8_va_mtl,mbf_va_mtl}/`.

In [1]:
import os, sys, subprocess, json
from pathlib import Path

REPO = Path.cwd().resolve().parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print('cwd:', Path.cwd())

cwd: C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code


## 1. Train `enet_b0_8_va_mtl`

Expected wall time: a few hours on GPU (EXPR 10 ep + VA 20 ep + AU 10 ep at batch 512).

In [ ]:
# Stream training logs into the notebook. Use -u to disable buffering so tqdm lines flush promptly.
cmd = [sys.executable, '-u', '-m', 'src.train', '--config', 'configs/aw2_stage1_enet.yaml']
print('$', ' '.join(cmd))
proc = subprocess.run(cmd, check=False)
assert proc.returncode == 0, f'src.train failed with exit code {proc.returncode}'

### Raw evaluation (no smoothing)

In [3]:
from src.eval import evaluate_mtl

ENET_CKPT = Path('results/aw2_stage1_enet/best.pt')
assert ENET_CKPT.exists(), f'missing checkpoint: {ENET_CKPT}'

enet_raw = evaluate_mtl(
    checkpoint=ENET_CKPT,
    config_path='configs/aw2_stage1_enet.yaml',
    smooth=False,
)

[stage1] metrics written to results\aw2_stage1_enet\metrics.md
           ccc_V = 0.4640
           ccc_A = 0.4211
          CCC_VA = 0.4425
   F1_EXPR_macro = 0.3369
        ACC_EXPR = 0.3575
       F1_AU@0.5 = 0.4796
      F1_AU_best = 0.4822
       t_AU_best = 0.6000
       P_MTL@0.5 = 1.2591
      P_MTL_best = 1.2616
       smoothing = 0.0000
           sigma = nan
           delta = nan


## 2. Smoothing grid — shared helper

`evaluate_mtl` reloads the checkpoint and re-runs the forward pass on every call. For a 9 × 5 grid that is wasteful, so the cell below caches predictions once per backbone and iterates over (σ, δ) applying only the smoothing + metrics step. The logic mirrors `src.eval.evaluate_mtl` exactly.

In [4]:
import numpy as np
import torch
from omegaconf import OmegaConf

from src.heads.mtl_head import MTLHead, MTLHeadConfig
from src.smoothing import gaussian_smooth_per_video
from src.train import load_split
from src.utils.metrics import (
    f1_macro_au, f1_score_max, metric_for_Exp, metric_for_VA, p_mtl,
)


def _forward_once(checkpoint, config_path):
    cfg = OmegaConf.load(config_path)
    device = cfg.train.device if torch.cuda.is_available() else 'cpu'
    X_val, anno_val = load_split(cfg.data.val_annotations, cfg.features_cache)

    ck = torch.load(checkpoint, map_location='cpu', weights_only=False)
    head_cfg = MTLHeadConfig(**ck['head_cfg'])
    model = MTLHead(head_cfg).to(device)
    model.load_state_dict(ck['state_dict'])
    model.eval()

    preds_expr, preds_va, preds_au = [], [], []
    with torch.no_grad():
        xt = torch.from_numpy(X_val.astype(np.float32))
        for i in range(0, xt.size(0), 4096):
            xb = xt[i:i + 4096].to(device)
            e, v, a = model(xb)
            preds_expr.append(torch.softmax(e, dim=-1).cpu().numpy())
            preds_va.append(v.cpu().numpy())
            preds_au.append(a.cpu().numpy())
    return (
        anno_val,
        np.concatenate(preds_expr, 0),
        np.concatenate(preds_va, 0),
        np.concatenate(preds_au, 0),
        head_cfg,
    )


def _metrics(anno, pred_expr_prob, pred_va, pred_au, head_cfg):
    va_m = anno.mask_va == 1
    ccc_V, ccc_A, ccc_VA = metric_for_VA(
        anno.y_va[va_m, 0], anno.y_va[va_m, 1],
        pred_va[va_m, 0], pred_va[va_m, 1],
    )
    ex_m = anno.mask_expr == 1
    y_pred_expr = pred_expr_prob.argmax(axis=1)
    f1_expr, _, _ = metric_for_Exp(
        anno.y_expr[ex_m], y_pred_expr[ex_m], class_num=head_cfg.num_expr,
    )
    au_m = anno.mask_au == 1
    f1_au = f1_macro_au(anno.y_aus[au_m], pred_au[au_m], threshold=0.5)
    return ccc_V, ccc_A, ccc_VA, f1_expr, f1_au, p_mtl(ccc_VA, f1_expr, f1_au)


def smoothing_grid(checkpoint, config_path,
                    sigmas=(0.1, 1, 10, 50, 100, 500, 1e3, 1e4, 1e5),
                    deltas=(1, 5, 10, 50, 100)):
    anno, pred_expr_prob, pred_va, pred_au, head_cfg = _forward_once(checkpoint, config_path)
    baseline = _metrics(anno, pred_expr_prob, pred_va, pred_au, head_cfg)
    rows = [('raw', None, None, *baseline)]
    best_p, best_key = baseline[-1], ('raw', None, None)
    for sigma in sigmas:
        for delta in deltas:
            sm_expr = gaussian_smooth_per_video(pred_expr_prob, anno.videoname_frames, sigma=sigma, delta=delta)
            sm_va   = gaussian_smooth_per_video(pred_va,        anno.videoname_frames, sigma=sigma, delta=delta)
            m = _metrics(anno, sm_expr, sm_va, pred_au, head_cfg)
            rows.append(('sm', sigma, delta, *m))
            if m[-1] > best_p:
                best_p = m[-1]
                best_key = ('sm', sigma, delta)
    return rows, best_key, best_p


### enet smoothing grid

In [5]:
enet_rows, enet_best_key, enet_best_p = smoothing_grid(ENET_CKPT, 'configs/aw2_stage1_enet.yaml')
print(f'enet  raw       P_MTL = {enet_rows[0][-1]:.4f}')
print(f'enet  best key  (sigma, delta) = ({enet_best_key[1]}, {enet_best_key[2]})  -> P_MTL = {enet_best_p:.4f}')
print(f'enet  delta     = +{enet_best_p - enet_rows[0][-1]:.4f}')

enet  raw       P_MTL = 1.2591
enet  best key  (sigma, delta) = (500, 10)  -> P_MTL = 1.3688
enet  delta     = +0.1097


## 3. Train `mbf_va_mtl`

In [ ]:
cmd = [sys.executable, '-u', '-m', 'src.train', '--config', 'configs/aw2_stage1_mbf.yaml']
print('$', ' '.join(cmd))
proc = subprocess.run(cmd, check=False)
assert proc.returncode == 0, f'src.train failed with exit code {proc.returncode}'

In [7]:
MBF_CKPT = Path('results/aw2_stage1_mbf/best.pt')
assert MBF_CKPT.exists(), f'missing checkpoint: {MBF_CKPT}'

mbf_raw = evaluate_mtl(
    checkpoint=MBF_CKPT,
    config_path='configs/aw2_stage1_mbf.yaml',
    smooth=False,
)
mbf_rows, mbf_best_key, mbf_best_p = smoothing_grid(MBF_CKPT, 'configs/aw2_stage1_mbf.yaml')
print(f'mbf   raw       P_MTL = {mbf_rows[0][-1]:.4f}')
print(f'mbf   best key  (sigma, delta) = ({mbf_best_key[1]}, {mbf_best_key[2]})  -> P_MTL = {mbf_best_p:.4f}')
print(f'mbf   delta     = +{mbf_best_p - mbf_rows[0][-1]:.4f}')

[stage1] metrics written to results\aw2_stage1_mbf\metrics.md
           ccc_V = 0.4697
           ccc_A = 0.4339
          CCC_VA = 0.4518
   F1_EXPR_macro = 0.2913
        ACC_EXPR = 0.3170
       F1_AU@0.5 = 0.4785
      F1_AU_best = 0.4819
       t_AU_best = 0.6000
       P_MTL@0.5 = 1.2217
      P_MTL_best = 1.2251
       smoothing = 0.0000
           sigma = nan
           delta = nan
mbf   raw       P_MTL = 1.2217
mbf   best key  (sigma, delta) = (1000.0, 10)  -> P_MTL = 1.3039
mbf   delta     = +0.0822


## 4. Reproduction summary

Writes a markdown table under `results/aw2_stage1/summary.md` with raw vs. best-smoothed numbers next to Paper A cell 60 'aligned' targets. The reproduction is deemed successful if `|P_MTL_ours - P_MTL_paperA| <= 0.015` on both backbones.

In [8]:
OUT = Path('results/aw2_stage1')
OUT.mkdir(parents=True, exist_ok=True)

# Paper A cell-60 aligned targets.
PAPER_A = {
    'enet_b0_8_va_mtl': dict(ccc_V=0.4433, ccc_A=0.3422, f1_expr=0.5040, p_mtl=1.2896),
    'mbf_va_mtl':        dict(ccc_V=0.4503, ccc_A=0.2870, f1_expr=0.4891, p_mtl=1.2264),
}

def row(backbone, rows_grid, best_key, best_p):
    raw = rows_grid[0]   # ('raw', None, None, ccc_V, ccc_A, ccc_VA, f1_expr, f1_au, p_mtl)
    tag, _sigma, _delta, ccc_V, ccc_A, ccc_VA, f1_expr, f1_au, p = raw
    t = PAPER_A[backbone]
    delta_p = p - t['p_mtl']
    within = abs(delta_p) <= 0.015
    return dict(
        backbone=backbone,
        raw=dict(ccc_V=ccc_V, ccc_A=ccc_A, f1_expr=f1_expr, f1_au=f1_au, p_mtl=p),
        best=dict(sigma=best_key[1], delta=best_key[2], p_mtl=best_p),
        target=t,
        delta_p=delta_p,
        within_gate=bool(within),
    )

summary = [
    row('enet_b0_8_va_mtl', enet_rows, enet_best_key, enet_best_p),
    row('mbf_va_mtl',        mbf_rows,  mbf_best_key,  mbf_best_p),
]

lines = []
lines.append('# Aff-Wild2 Stage 1 reproduction summary\n')
lines.append('')
lines.append('## Raw (no smoothing) vs. Paper A cell 60 "aligned" targets\n')
lines.append('| backbone | CCC_V (ours / paper) | CCC_A | F1_EXPR | P_MTL (ours / paper) | delta_P_MTL | within +/-0.015? |')
lines.append('| --- | --- | --- | --- | --- | --- | --- |')
for s in summary:
    b = s['backbone']; r = s['raw']; t = s['target']
    lines.append(
        f"| `{b}` | {r['ccc_V']:.4f} / {t['ccc_V']:.4f} | {r['ccc_A']:.4f} / {t['ccc_A']:.4f} | "
        f"{r['f1_expr']:.4f} / {t['f1_expr']:.4f} | {r['p_mtl']:.4f} / {t['p_mtl']:.4f} | "
        f"{s['delta_p']:+.4f} | {'yes' if s['within_gate'] else 'NO'} |"
    )
lines.append('')
lines.append('## Smoothing grid — best (sigma, delta) per backbone\n')
lines.append('| backbone | sigma* | delta* | P_MTL smoothed | delta_over_raw |')
lines.append('| --- | --- | --- | --- | --- |')
for s in summary:
    lines.append(
        f"| `{s['backbone']}` | {s['best']['sigma']} | {s['best']['delta']} | {s['best']['p_mtl']:.4f} | "
        f"+{s['best']['p_mtl'] - s['raw']['p_mtl']:.4f} |"
    )
lines.append('')

(OUT / 'summary.md').write_text('\n'.join(lines), encoding='utf-8')
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print((OUT / 'summary.md').read_text(encoding='utf-8'))

# Aff-Wild2 Stage 1 reproduction summary


## Raw (no smoothing) vs. Paper A cell 60 "aligned" targets

| backbone | CCC_V (ours / paper) | CCC_A | F1_EXPR | P_MTL (ours / paper) | delta_P_MTL | within +/-0.015? |
| --- | --- | --- | --- | --- | --- | --- |
| `enet_b0_8_va_mtl` | 0.4640 / 0.4433 | 0.4211 / 0.3422 | 0.3369 / 0.5040 | 1.2591 / 1.2896 | -0.0305 | NO |
| `mbf_va_mtl` | 0.4697 / 0.4503 | 0.4339 / 0.2870 | 0.2913 / 0.4891 | 1.2217 / 1.2264 | -0.0047 | yes |

## Smoothing grid — best (sigma, delta) per backbone

| backbone | sigma* | delta* | P_MTL smoothed | delta_over_raw |
| --- | --- | --- | --- | --- |
| `enet_b0_8_va_mtl` | 500 | 10 | 1.3688 | +0.1097 |
| `mbf_va_mtl` | 1000.0 | 10 | 1.3039 | +0.0822 |



Drop the `summary.md` contents into Section 6.x (`tab:stage1aw2_targets` neighbour) of `internship_report.tex`; both rows should land within ±0.015 P_MTL of Paper A. If they do not, see `affwild2_coding_plan.md` §3 for the suspect-fix order.